# miniLISA TDI Noise Modelling

Symbolic modelling of the miniLISA lab testbed described in the article draft. Three simulated spacecraft exchange heterodyne beatnotes via digital delay lines. This notebook models the one-way phase measurements (carrier and sidebands), constructs first-generation TDI combinations (X1 and Sagnac α₁), and applies clock-noise corrections.

**Noise terms included via toggle flags** at the bottom of the setup cell.

In [31]:
from sympy import Function, Symbol, symbols, simplify, latex, collect, expand
from IPython.display import display, Math
import re

## Utilities

In [32]:
def color_terms(latex_str):
    """Color-code noise terms in LaTeX output for readability."""
    # REF laser phase → purple
    latex_str = re.sub(
        r'(\\phi_\{REF\}\{\\left\(.+?\\right\)\})',
        r'{\\color{purple} \1}', latex_str)
    # q_1 (Moku clock, SC1) → red
    latex_str = re.sub(
        r'(q_\{([1])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{red} \1{\\left(\3\\right)}}', latex_str)
    # q_2 → orange
    latex_str = re.sub(
        r'(q_\{([2])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}', latex_str)
    # q_3 → yellow
    latex_str = re.sub(
        r'(q_\{([3])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{yellow} \1{\\left(\3\\right)}}', latex_str)
    # epsilon_A (delay-board clock, SC1) → cyan
    latex_str = re.sub(
        r'(\\epsilon_\{([A])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Cyan} \1{\\left(\3\\right)}}', latex_str)
    # epsilon_B → aquamarine
    latex_str = re.sub(
        r'(\\epsilon_\{([B])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Aquamarine} \1{\\left(\3\\right)}}', latex_str)
    # epsilon_C → spring green
    latex_str = re.sub(
        r'(\\epsilon_\{([C])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{SpringGreen} \1{\\left(\3\\right)}}', latex_str)
    return latex_str

def show(label, expr):
    display(Math(label + color_terms(latex(expr))))

In [33]:
# ── Time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── Delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True)

# ── Carrier frequencies ───────────────────────────────────────────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)

# ── Modulation frequencies ────────────────────────────────────────────────────
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)
omega_rm = Symbol(r'\omega_r^m', real=True)   # shared reference modulation freq.

# ── Laser phase noise ─────────────────────────────────────────────────────────
phi1  = Function(r'\phi_1')
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')
phiREF = Function(r'\phi_{REF}')   # shared reference laser phase

# ── Moku phasemeter clock noise (LISA-like, referenced to global time) ────────
q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')
q_ref = Function('q_{ref}')        # reference phasemeter clock

# ── Delay-board clock noise (ε variables, one board per spacecraft) ───────────
epsilonA = Function(r'\epsilon_A')   # board A → handles delays into SC1
epsilonB = Function(r'\epsilon_B')   # board B → delays into SC2
epsilonC = Function(r'\epsilon_C')   # board C → delays into SC3

# ── Optical path noise (phasemeter readout noise per link) ───────────────────
#   N_{i_j}: carrier readout noise measured on board i from laser j
#   n_{i_j}: sideband readout noise
N1_1 = Function('N_{1_1}'); N1_2 = Function('N_{1_2}'); N1_3 = Function('N_{1_3}')
N2_1 = Function('N_{2_1}'); N2_2 = Function('N_{2_2}'); N2_3 = Function('N_{2_3}')
N3_1 = Function('N_{3_1}'); N3_2 = Function('N_{3_2}'); N3_3 = Function('N_{3_3}')
n1_1 = Function('n_{1_1}'); n1_2 = Function('n_{1_2}'); n1_3 = Function('n_{1_3}')
n2_1 = Function('n_{2_1}'); n2_2 = Function('n_{2_2}'); n2_3 = Function('n_{2_3}')
n3_1 = Function('n_{3_1}'); n3_2 = Function('n_{3_2}'); n3_3 = Function('n_{3_3}')

# ── Modulation (EOM) phase noise ──────────────────────────────────────────────
N1_m = Function('P_{1}^{m}')
N2_m = Function('P_{2}^{m}')
N3_m = Function('P_{3}^{m}')


# q_ref: reference laser timing variable
q_ref = Function('q_{ref}')

In [34]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon, N_mod)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, N1_m),
    2: (phi2, q2, omega2, omega2m, epsilonB, N2_m),
    3: (phi3, q3, omega3, omega3m, epsilonC, N3_m),
}

tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

# Optical readout noise: (carrier_local, carrier_incoming, sb_local, sb_incoming)
N_noise = {
    (1,2): (N1_1, N1_2, n1_1, n1_2), (1,3): (N1_1, N1_3, n1_1, n1_3),
    (2,1): (N2_1, N2_2, n2_1, n2_2), (2,3): (N2_2, N2_3, n2_2, n2_3),
    (3,1): (N3_1, N3_3, n3_1, n3_2), (3,2): (N3_3, N3_2, n3_3, n3_2),
}

In [35]:
def D(expr, tau_val):
    """Apply delay operator: D(x(t), τ) = x(t - τ)."""
    return expr.subs(t, t - tau_val)

In [36]:
# ── Noise-term toggles ─────────────────────────────────────────────────────
include_phi           = False   # Laser phase noise φ_i
include_clock_noise   = True    # Moku phasemeter clock noise q_i (LISA-like)
include_board_jitter  = True   # Delay-board clock noise ε_i
include_REF_laser     = False   # Shared reference laser φ_REF
include_optical_noise = False    # Phasemeter readout / optical path noise N
include_modulation_noise = False # EOM modulation phase noise P^m
include_ref_mod       = True    # Reference-modulation sideband coupling (ω_r^m)

# ── Build all six one-way phase measurements ────────────────────────────────
eta     = {}   # carrier
etaSB   = {}   # upper sideband
etaLSB  = {}   # lower sideband
REF     = {}   # reference interferometer variable (used for board-jitter correction)
r       = {}   # clock correcting variable r_{ij} / ω_j^m

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i, N_i_m = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, N_j_m = sc[j]
    t_ij = tau[(i, j)]
    Nij, Nji, nij, nji = N_noise[(i, j)]

    # Laser phase terms (with optional common reference laser)
    phi_terms = (
        D(phi_j(t) - int(include_REF_laser)*phiREF(t), t_ij)
        - (phi_i(t) - int(include_REF_laser)*phiREF(t))
    ) if include_phi else 0

    # Moku clock noise — carrier: couples as (ω_j − ω_i)·q_i(t)
    clock_C   = (-(om_j - om_i) * q_i(t)) if include_clock_noise else 0

    # Moku clock noise — upper sideband: full coupling including modulation offset
    clock_SB  = (-(om_j - om_i + omm_j - omm_i) * q_i(t)
                 - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0

    # Moku clock noise — lower sideband (sign flip on modulation part)
    clock_LSB = (-( -(om_i - om_j + omm_j - omm_i) * q_i(t)
                 - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij))) if include_clock_noise else 0

    # Delay-board jitter ε_i — couples via ω_j * (ε_i(t) − ε_i(t−τ_{ij}))
    board_C   = (om_j * (eps_i(t) - D(eps_i(t), t_ij)) ) if include_board_jitter else 0
    board_SB  = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    board_LSB  = ((om_j - omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0

    # Optical / readout noise
    opt_C     = (Nij(t) - D(Nji(t), t_ij)) if include_optical_noise else 0
    opt_SB    = (nij(t) - D(nji(t), t_ij)) if include_optical_noise else 0

    # EOM modulation phase noise
    mod_noise = (N_i_m(t) - D(N_j_m(t), t_ij)) if include_modulation_noise else 0

    # Reference-modulation coupling: ω_r^m·(q_j(t−τ) − q_i(t))
    # This arises because the reference laser sideband modulation is
    # synthesised from a common reference oscillator (ω_r^m), so its
    # timing jitter appears in both the sent and locally received sideband.
    ref_mod_SB  =  -omega_rm * (D(q_ref(t), t_ij) - q_ref(t)) + int(include_board_jitter)*omega_rm * (D(eps_i(t), t_ij) - eps_i(t)) if include_ref_mod else 0
    ref_mod_LSB = omega_rm * (D(q_ref(t), t_ij) - q_ref(t)) - int(include_board_jitter)*omega_rm * (D(eps_i(t), t_ij) - eps_i(t)) if include_ref_mod else 0

    # Assemble measurements
    eta[(i,j)]    = phi_terms + clock_C + board_C + opt_C

    etaSB[(i,j)]  = collect(expand(
        phi_terms + clock_SB + board_SB + opt_SB + mod_noise + ref_mod_SB
    ), [q_i(t), q_j(t), eps_i(t), D(q_j(t), t_ij), D(eps_i(t), t_ij)])

    etaLSB[(i,j)] = collect(expand(
        phi_terms + clock_LSB + board_LSB + opt_SB + mod_noise + ref_mod_LSB
    ), [q_i(t), q_j(t), eps_i(t), D(q_j(t), t_ij), D(eps_i(t), t_ij)])
    
    # REF variable (for delay-board jitter correction via reference interferometer)
    REF[(i,j)] = (
        -int(include_clock_noise) * q_j(t)
        + int(include_board_jitter) * eps_i(t)
    )

    # Clock-correcting variable r_{ij}: formed from sideband difference
    # r_{ij} = -(η^{LSB}_{ij} − η^{SB}_{ij}) / (2·(ω_j^m + ω_r^m))
    r[(i,j)] = simplify(-(etaLSB[(i,j)] - etaSB[(i,j)]) / omm_j  / 2)

In [37]:
for (i, j) in tau:
    show(rf'\eta_{{{i}{j}}} = ', eta[(i,j)])
    show(rf'\eta^{{SB}}_{{{i}{j}}} = ', etaSB[(i,j)])
    show(rf'\eta^{{LSB}}_{{{i}{j}}} = ', etaLSB[(i,j)])
    show(rf'r_{{{i}{j}}} / \omega_{j}^m = ', r[(i,j)])
    print('─' * 60)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

────────────────────────────────────────────────────────────


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

────────────────────────────────────────────────────────────


## First-generation TDI — X₁ Michelson combination

The X₁ combination on spacecraft 1 is:

$$X_1 = P_{13}(\eta_{13} + D_{13}\,\eta_{31}) + P_{12}(\eta_{12} + D_{12}\,\eta_{21})$$

where the path operators P_{1k} time-shift and difference each arm so that laser noise from SC2 and SC3 cancels (in the equal-arm, static-delay limit).

In [38]:
def P12(expr): return expr - D(expr, tau13 + tau31)
def P21(expr): return D(expr, tau12) - D(expr, tau12 + tau13 + tau31)
def P13(expr): return -(expr - D(expr, tau12 + tau21))
def P31(expr): return -(D(expr, tau13) - D(expr, tau13 + tau12 + tau21))

collect_vars = [q1(t), q2(t), q3(t), epsilonA(t), epsilonB(t), epsilonC(t),
                phi1(t), phi2(t), phi3(t)]

X1 = collect(expand(
    P13(eta[(1,3)] + D(eta[(3,1)], tau13)) +
    P12(eta[(1,2)] + D(eta[(2,1)], tau12))
), collect_vars)

show(r'X_1 = ', X1)

<IPython.core.display.Math object>

### Clock-noise correction for X₁

Following Vallisneri's algorithm, the clock-jitter correction term is assembled from
the correcting variables R_{ij}, which are delayed combinations of r_{ij}:

$$X_1^{\rm corr} = X_1 - \sum_{(i,j,k)} \left[ (\omega_i-\omega_j)R_{ij} + (\omega_i-\omega_k)R_{ik}\right]$$

In [39]:
# ── Correcting variables R_{ij} ─────────────────────────────────────────────
R = {}
R[(1,2)] = -(r[(1,3)] + D(r[(3,1)], tau13))
R[(1,3)] =   r[(1,2)] + D(r[(2,1)], tau12)
R[(2,1)] = (  r[(1,2)] - r[(1,3)]
             - D(r[(3,1)], tau13) - D(r[(1,2)], tau13 + tau31))
R[(3,1)] = ( -r[(1,3)] + r[(1,2)]
             + D(r[(2,1)], tau12) + D(r[(1,3)], tau12 + tau21))
R[(2,3)] = 0
R[(3,2)] = 0

a = {
    (1,2): omega1 - omega2, (2,1): omega2 - omega1,
    (1,3): omega1 - omega3, (3,1): omega3 - omega1,
    (2,3): omega2 - omega3, (3,2): omega3 - omega2,
}

triplets = [(1,2,3), (2,3,1), (3,1,2)]
correction = 0
for (i, j, k) in triplets:
    correction -= (-a[(i,j)] * R[(i,j)] - a[(i,k)] * R[(i,k)])

X1c = simplify(X1 - correction)

print("X1 (uncorrected):")
show(r'X_1 = ', X1)
print("\nX1 (clock-noise corrected):")

temp = X1c*(omega2m*omega3m + omega2m*omega_rm + omega3m*omega_rm + omega_rm*omega_rm)

show(r'X_1^{\rm corr} = ', collect(expand(temp), [omega1*omega2m*omega3m, omega1*omega2m*omega_rm, omega1*omega3m*omega_rm, omega1*omega_rm*omega_rm, omega1*omega_rm*omega_rm, omega2*omega3m*omega_rm, omega2*omega_rm*omega_rm, omega2*omega_rm*omega_rm, omega2*omega_rm*omega_rm, omega2*omega_rm*omega_rm, omega2*omega_rm*omega_rm, omega2*omega_rm*omega_rm, omega3*omega2m*omega_rm, omega3*omega_rm*omega_rm]))

X1 (uncorrected):


<IPython.core.display.Math object>


X1 (clock-noise corrected):


<IPython.core.display.Math object>

### Delay-board jitter correction via reference interferometer

The delay-line board jitter ε_i can be further corrected using cross-spacecraft REF measurements.
The REF_AB, REF_AC, REF_CB variables are formed from pairs of reference-interferometer readouts.

In [50]:
REF_AB = REF[(1,3)] - REF[(2,3)]
REF_AC = REF[(3,2)] - REF[(1,2)]
REF_CB = REF[(2,1)] - REF[(3,1)]

show(r'\mathrm{REF}_{AB} = ', REF_AB)
show(r'\mathrm{REF}_{AC} = ', REF_AC)
show(r'\mathrm{REF}_{CB} = ', REF_CB)

Corr1 = (omega1*omega2m*omega3m + omega1*omega2m*omega_rm + omega1*omega3m*omega_rm + omega1*omega_rm*omega_rm) * (D(REF_AB, tau12) - D(REF_AB, tau12 + tau21) - D(REF_AB, tau12 + tau13 + tau31))
Corr2 = (omega1*omega2m*omega3m + omega1*omega2m*omega_rm + omega1*omega3m*omega_rm + omega1*omega_rm*omega_rm) * (D(REF_AC, tau13) - D(REF_AC, tau13 + tau31) - D(REF_AC, tau12 + tau13 + tau21))
Corr3 = -(omega1*omega2m*omega3m + omega1*omega2m*omega_rm + omega1*omega3m*omega_rm + omega1*omega_rm*omega_rm) * D(REF_CB, tau12 + tau13 + tau21 + tau31)

X1_fully_corrected = simplify(X1c* (omega2m*omega3m + omega2m*omega_rm + omega3m*omega_rm + omega_rm*omega_rm)  + Corr1 + Corr2 + Corr3)
print("\nX1 after board-jitter REF correction:")
expr = X1_fully_corrected *2 *omega2m*omega3m  / (omega2m*omega3m + omega2m*omega_rm + omega3m*omega_rm + omega_rm*omega_rm) / (2*omega_rm)
show(r'X_1^{\rm full\ corr} = ', simplify(expr))

show(r'X_1^{\rm full\ corr} = ', collect(expand(expr), [omega1*omega2m, omega1*omega3m, omega1*omega_rm, omega2*omega3m, omega2*omega_rm, omega3*omega2m, omega3*omega_rm]))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


X1 after board-jitter REF correction:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [41]:
# Second layer bc why not
Corr_AtoB = omega1*omega2m*(REF_AB - D(REF_AB, tau13) - D(REF_AB, tau12 + tau21) + D(REF_AB, tau12 + tau13 + tau21))
show(r'test = ', simplify(Corr_AtoB))

<IPython.core.display.Math object>

In [42]:
show(r'X_1^{\rm full\ corr} = ', collect(expand(expr-Corr_AtoB), [omega1*omega2m, omega1*omega3m, omega1*omega_rm, omega2*omega3m, omega2*omega_rm, omega3*omega2m, omega3*omega_rm]))

<IPython.core.display.Math object>

In [43]:
Corr_AtoB2 = (omega1*omega2m+omega2m*omega3)*(REF_AB - D(REF_AB, tau13) - D(REF_AB, tau12 + tau21) + D(REF_AB, tau12 + tau13 + tau21))
show(r'X_1^{\rm full\ corr} = ', collect(expand(expr-Corr_AtoB+Corr_AtoB2), [omega1*omega2m, omega1*omega3m, omega1*omega_rm, omega2*omega3m, omega2*omega_rm, omega3*omega2m, omega3*omega_rm]))

<IPython.core.display.Math object>

## First-generation TDI — Sagnac α₁ combination

The Sagnac combination α₁ centred on SC1 uses all six one-way links in a clockwise/counter-clockwise pair:

$$\alpha_1 = \eta_{12} + D_{12}\,\eta_{23} + D_{123}\,\eta_{31} - \eta_{13} - D_{13}\,\eta_{32} - D_{132}\,\eta_{21}$$

In [44]:
def P12s(expr): return expr
def P23s(expr): return D(expr, tau12)
def P31s(expr): return D(expr, tau12 + tau23)
def P13s(expr): return -expr
def P32s(expr): return -D(expr, tau13)
def P21s(expr): return -D(expr, tau13 + tau32)

R_sagnac = {}
R_sagnac[(1,2)] = 0
R_sagnac[(2,3)] = r[(1,2)]
R_sagnac[(3,1)] = r[(1,2)] + D(r[(2,3)], tau12)
R_sagnac[(1,3)] = 0
R_sagnac[(3,2)] = -r[(1,3)]
R_sagnac[(2,1)] = -(r[(1,3)] + D(r[(3,2)], tau13))

sagnac_correction = 0
for (i, j, k) in triplets:
    sagnac_correction -= (-a[(i,j)] * R_sagnac[(i,j)] - a[(i,k)] * R_sagnac[(i,k)])

alpha1 = collect(expand(
    P12s(eta[(1,2)]) + P23s(eta[(2,3)]) + P31s(eta[(3,1)])
  + P13s(eta[(1,3)]) + P32s(eta[(3,2)]) + P21s(eta[(2,1)])
), collect_vars)

alpha1c = simplify(alpha1 - sagnac_correction)

print("α₁ (uncorrected):")
show(r'\alpha_1 = ', alpha1)
print("\nα₁ (clock-noise corrected):")
show(r'\alpha_1^{\rm corr} = ', alpha1c)

α₁ (uncorrected):


<IPython.core.display.Math object>


α₁ (clock-noise corrected):


<IPython.core.display.Math object>

### Sagnac board-jitter REF correction

In [45]:
CorrA_sagnac  = omega1 * (D(REF_AC, tau13) + D(REF_AB, tau12))
CorrBC_sagnac = omega1 * (
      D(REF_CB, tau12 + tau23)
    + D(REF_CB, tau13 + tau32)
    - D(REF_CB, tau12 + tau23 + tau31)
)

alpha1_fully_corrected = simplify(alpha1c + CorrA_sagnac + CorrBC_sagnac).subs(
    {tau12: tau21, tau13: tau31, tau23: tau32})
print("α₁ after board-jitter REF correction:")
show(r'\alpha_1^{\rm full\ corr} = ', alpha1_fully_corrected)

α₁ after board-jitter REF correction:


<IPython.core.display.Math object>